![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 28 -- Competition: Camera or Generated? (starter)

A photo-sharing app receives thousands of tiny thumbnails every day. Some are real photos taken with a camera. Others were made by an image-generation model that was trained to imitate such photos, and the app wants to flag those automatically. For the training pictures you get the picture and the answer; for the test pictures you get only the picture.

**Your task:** decide, for every test picture, whether it was made by the model (`generated` = 1) or taken with a camera (`generated` = 0). Exactly half of the pictures are generated, in train and in test.

**Your role:** start from this notebook. Keep the two cells marked *do not modify* as they are, change anything else, and upload the `submission.csv` this notebook writes to the Kaggle page. Hand in your final notebook through the Day 28 form.

**Dataset:** `train/camera/` and `train/generated/` (3,500 pictures each, 32 x 32 pixels, RGB), `train.csv` (one row per training picture: `id`, `generated`, `path`), `test/` (3,000 pictures named `<id>.png`) and `sample_submission.csv` (the exact shape of the file you upload: `id,generated`, all 0). The pictures are described in the Kaggle Data tab, in the competition README and in the cell below.

## Rules in short

- Teams of at most 3 (solo is fine); 10 submissions per team; choose your 2 final submissions before the end.
- The score is **accuracy**: the share of test pictures you label correctly. The public leaderboard uses 40% of the test pictures; the final ranking uses the other 60%.
- Allowed: PyTorch, torchvision, NumPy, pandas, matplotlib, scikit-learn (anything taught in Weeks 3 and 4). Every parameter of your model starts random: no pretrained weights. No other pictures, no searching for where these pictures come from.
- Upload your final notebook through the Day 28 Google Form by 22:30 Riyadh time, one per team.

---
# Setup

In [ ]:
# torch, torchvision, numpy, pandas, matplotlib, scikit-learn and PIL are preinstalled on Kaggle and Colab. kagglehub is only needed outside Kaggle.
# !pip install -q kagglehub

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# ============================================================
#  DATA LOADING -- do not modify
# ============================================================
DATA_ID = "sattamjaltwaim/ai-olympiad-l1-2027-week-4-competition-data"          # the Kaggle dataset of this competition


def find_data():
    local = os.environ.get("COMP_DATA")                      # a local folder (instructor's check)
    if local and os.path.exists(os.path.join(local, "train.csv")):
        return local
    for folder in sorted(glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*")):   # inside a Kaggle notebook
        if os.path.exists(os.path.join(folder, "train.csv")):
            return folder
    import kagglehub                                         # Colab or your own computer
    return kagglehub.dataset_download(DATA_ID)


DATA = find_data()
TRAIN_DIR, TEST_DIR = os.path.join(DATA, "train"), os.path.join(DATA, "test")
train_df = pd.read_csv(os.path.join(DATA, "train.csv"))
sample = pd.read_csv(os.path.join(DATA, "sample_submission.csv"))
test_paths = [os.path.join(TEST_DIR, f"{i}.png") for i in sample["id"]]      # one path per test id, in the order of the sample file
print("training pictures:", len(train_df), "| test pictures:", len(test_paths))
print("share of generated == 1 in train:", round(train_df["generated"].mean(), 3))
print("folders under train/:", sorted(os.listdir(TRAIN_DIR)))

## The pictures, in plain words

Every picture is 32 x 32 pixels in colour (RGB), saved as a PNG. The training set has 3,500 camera pictures in `train/camera/` and 3,500 generated pictures in `train/generated/`; `train.csv` lists each one with its `id`, its `generated` label and its `path`. The test pictures sit in one folder, `test/`, named `<id>.png`, and `sample_submission.csv` lists their ids.

Both classes show the same kinds of things: everyday objects, vehicles and animals. The generator was trained to imitate exactly these kinds of pictures, so the difference is not *what* is in the picture but *how it was made*. Look at textures, edges, colour transitions and backgrounds; a convolutional network can learn such patterns from the pixels, which is what this week was about.

---
# Look at the pictures first

The Day 27 tip: look at a training batch before the first epoch. Eight pictures of each class, then the table that lists them. If you cannot tell them apart by eye, that is normal; the network looks at patterns you do not notice.

In [ ]:
print(train_df["generated"].value_counts().to_dict())
train_df.head()

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for row, cls in enumerate(["camera", "generated"]):
    paths = train_df[train_df["path"].str.contains(f"/{cls}/")]["path"].head(8)
    for ax, p in zip(axes[row], paths):
        ax.imshow(Image.open(os.path.join(DATA, p)))
        ax.set_title(cls, fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

---
# Datasets and loaders

`ImageFolder` turns the two folder names into class indices (Day 22, Lab 2). The folders are named so that the alphabetical order matches the label: `camera` is 0 and `generated` is 1. One in five training pictures is kept for validation; the test pictures come through a small `Dataset` class that also returns each picture's id, in the order of `sample_submission.csv`.

In [ ]:
transform = transforms.ToTensor()                      # 0..255 -> 0..1, shape (3, 32, 32)
full = datasets.ImageFolder(TRAIN_DIR, transform=transform)
print(full.class_to_idx)
assert full.class_to_idx == {"camera": 0, "generated": 1}, "class index must equal the generated label"

n_val = len(full) // 5
train_set, val_set = random_split(full, [len(full) - n_val, n_val], generator=torch.Generator().manual_seed(0))
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256)
xb, yb = next(iter(train_loader))
print(len(train_set), len(val_set), xb.shape, yb[:10].tolist())

In [ ]:
class TestPictures(Dataset):
    """The unlabelled test folder: returns (tensor, id)."""
    def __init__(self, paths, transform):
        self.paths, self.transform = paths, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), int(os.path.basename(self.paths[i])[:-4])


test_loader = DataLoader(TestPictures(test_paths, transform), batch_size=256)
xt, it = next(iter(test_loader))
print(xt.shape, it[:5].tolist())

---
# The model as a class

Day 22's plain CNN: three blocks of convolution, ReLU and max pooling take a 32 x 32 picture to 64 maps of 4 x 4, and one linear layer turns the 1,024 numbers into two scores, one per class. Shapes in the comments, the way Day 25 traces them.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # (16, 16, 16)
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # (32, 8, 8)
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))   # (64, 4, 4)
        self.classifier = nn.Linear(64 * 4 * 4, 2)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)


model = Net().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model(xb.to(device)).shape, "parameters:", n_params)

---
# The training loop

The four moves per batch (predict, loss, gradient, update), then a validation pass, then a checkpoint when the validation accuracy improves (Day 22, Lab 2). Eight epochs take about a minute on a GPU. This is the floor: the baseline scores about 0.72 to 0.75 on the test pictures; everything you learned this week is for beating it.

In [ ]:
EPOCHS = 8
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
history, best_acc = [], 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)        # 1. predict  2. loss
        opt.zero_grad(); loss.backward()     # 3. gradient
        opt.step()                           # 4. update
        train_loss += loss.item() * len(yb)
    model.eval()
    val_loss, correct = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            val_loss += loss_fn(logits, yb).item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
    train_loss, val_loss, val_acc = train_loss / len(train_set), val_loss / len(val_set), correct / len(val_set)
    history.append((train_loss, val_loss, val_acc))
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best.pt")        # the checkpoint of the best epoch
    print(f"epoch {epoch}: train loss {train_loss:.3f}, val loss {val_loss:.3f}, val accuracy {val_acc:.3f}")
print("best validation accuracy:", round(best_acc, 3))

## Read the curves

Training loss falling while validation loss rises is overfitting (Day 23, Lab 2); both falling together means there is more to learn.

In [ ]:
h = np.array(history)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(h[:, 0], label="train loss"); ax[0].plot(h[:, 1], label="val loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(h[:, 2], label="val accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout()
plt.show()

## The confusion matrix on the validation pictures

Rows are the true class, columns the predicted class. A model that calls everything `camera` would fill one column; check that both diagonal cells are large.

In [ ]:
best = Net().to(device)
best.load_state_dict(torch.load("best.pt", map_location=device))
best.eval()
preds, labels = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        preds.append(best(xb.to(device)).argmax(1).cpu()); labels.append(yb)
pred_val, y_val = torch.cat(preds).numpy(), torch.cat(labels).numpy()
print("validation accuracy of the checkpoint:", round(float((pred_val == y_val).mean()), 3))
print(pd.DataFrame(confusion_matrix(y_val, pred_val), index=["true camera", "true generated"], columns=["pred camera", "pred generated"]))

---
# Predict the test pictures

The same transform as in training (Day 27's mistake to avoid: one preprocessing for training, another for the test), the best checkpoint, `argmax` over the two scores. The class index is the label, because `camera` is 0 and `generated` is 1.

In [ ]:
test_ids, test_pred = [], []
with torch.no_grad():
    for xb, ib in test_loader:
        test_pred.append(best(xb.to(device)).argmax(1).cpu()); test_ids.append(ib)
test_ids, test_pred = torch.cat(test_ids).numpy(), torch.cat(test_pred).numpy()
print(len(test_ids), "test predictions; share predicted generated:", round(float(test_pred.mean()), 3))

---
# Write the submission

Pass the test ids and one 0/1 label per id. **Do not modify** the function.

In [ ]:
# ============================================================
#  SUBMISSION GENERATOR -- do not modify
# ============================================================
def make_submission(ids, generated, filename="submission.csv"):
    ids, generated = np.asarray(ids).astype(int), np.asarray(generated).astype(int)
    assert len(ids) == len(generated) == len(sample), f"{len(generated)} predictions for {len(sample)} test pictures"
    assert set(ids) == set(sample["id"]), "the ids must be exactly the test ids of sample_submission.csv"
    assert set(np.unique(generated)) <= {0, 1}, "predictions must be 0 or 1 (labels, not probabilities)"
    out = pd.DataFrame({"id": ids, "generated": generated}).set_index("id").loc[sample["id"]].reset_index()
    out.to_csv(filename, index=False)
    print(f"wrote {filename}: {len(out)} rows, share predicted generated = {out['generated'].mean():.3f}")
    return out


submission = make_submission(test_ids, test_pred)
submission.head()

---
# Where the points are

- **Look at a training batch before the first epoch, and make a valid file in the first twenty minutes** (Day 27, Tips). Then change one thing per run and log every run.
- **Augment, in the training transform only.** Horizontal flips and small random crops are label-safe here: nothing about *how a picture was made* depends on left or right (Day 23, Augmentation; Day 27, "augment with the defaults, then move the dials"). Keep the validation and test transforms plain.
- **Go VGG-style.** Two 3 x 3 convolutions per block, `BatchNorm2d` after each, three stages of 32, 64 and 128 channels, then a small head (Day 23, The Recipe and Architectures). Day 27's table of measured gains says what each change is worth.
- **Train longer, with a schedule, and keep the best checkpoint by validation accuracy** (Day 17; Day 23, Lab 2). Twenty to forty epochs of a small network cost a few minutes on a GPU.
- **Normalise the inputs** with the mean and standard deviation of the training pictures, the same numbers for validation and test.
- **Test-time augmentation:** average the probabilities of a picture and its mirror image, after measuring on validation that it helps (Day 27, Tricks).
- **Average several models' probabilities** (different seeds, or the best checkpoints of two runs); then take `argmax` (Day 27).
- **Trust your validation over the public leaderboard.** It scores 40% of the test pictures; the other 60% decide.

---

Made By **Sattam Altwaim**